In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import sys
import re
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path(r"d:/sugarcare_backend")
DATA_DIR = BASE_DIR / "data" / "your_datasets"
OUTPUT_DIR = BASE_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE_DIR / "data"))
from food_image_urls import get_food_image_url  # type: ignore

print("=" * 70)
print("DIABETIC MEAL RECOMMENDATION SYSTEM")
print("Personalized recommendations based on glucose, control level & time")
print("=" * 70)

In [ ]:
print("\n[STEP 1] Loading Datasets...")
print("-" * 50)

MEAL_CSV = DATA_DIR / "meal_recommender.csv"
meal_df = pd.read_csv(MEAL_CSV)
print(f"1. Meal Nutrition Data: {len(meal_df)} foods")
print(f"   Columns: {list(meal_df.columns)}")

DIABETES_CSV = DATA_DIR / "diabetes_dataset.csv"
diabetes_patient_df = pd.read_csv(DIABETES_CSV)
print(f"\n2. Diabetes Patient Data: {len(diabetes_patient_df)} records")
print(f"   Columns: {list(diabetes_patient_df.columns)}")

HEALTH_CSV = DATA_DIR / "diabetes_012_health_indicators_BRFSS2015.csv"
health_df = pd.read_csv(HEALTH_CSV)
print(f"\n3. Health Indicators: {len(health_df)} records")
print(f"   Columns: {list(health_df.columns)[:10]}...")

In [ ]:
print("\n[STEP 2] Feature Encodings...")
print("-" * 50)

CONTROL_LEVEL_ENCODING = {
    "well controlled": 0,
    "moderately controlled": 1,
    "poorly controlled": 2
}

PORTION_SIZE_ENCODING = {
    "small": 0,
    "medium": 1,
    "large": 2
}

PORTION_MULTIPLIERS = {
    "small": 0.75,
    "medium": 1.0,
    "large": 1.25
}

MEAL_TYPE_ENCODING = {
    "breakfast": 0,
    "lunch": 1,
    "dinner": 2,
    "snacks": 3
}

def time_to_meal_type(time_str: str) -> str:
    """Convert time string (HH:MM) to meal type."""
    try:
        hour = int(time_str.split(":")[0])
        if 5 <= hour < 11:
            return "breakfast"
        elif 11 <= hour < 15:
            return "lunch"
        elif 15 <= hour < 18:
            return "snacks"
        elif 18 <= hour < 22:
            return "dinner"
        else:
            return "snacks"
    except:
        return "lunch"

print("Control Level Encoding:", CONTROL_LEVEL_ENCODING)
print("Portion Size Encoding:", PORTION_SIZE_ENCODING)
print("Meal Type Encoding:", MEAL_TYPE_ENCODING)
print("\nTime → Meal Type Examples:")
for t in ["07:30", "12:30", "16:00", "19:30"]:
    print(f"   {t} → {time_to_meal_type(t)}")

In [ ]:
print("\n[STEP 3] Building Nutrition Database...")
print("-" * 50)

def safe_float(val, default=0.0):
    try:
        return float(val) if pd.notna(val) else default
    except:
        return default

nutrition_map = {}
gi_map = {}
meal_image_map = {}

for _, row in meal_df.iterrows():
    food_name = str(row['Food Name']).strip()
    if not food_name or food_name == 'nan':
        continue
    
    gi = safe_float(row.get('Glycemic Index', 0))
    carbs = safe_float(row.get('Carbohydrates', 0))
    protein = safe_float(row.get('Protein', 0))
    fat = safe_float(row.get('Fat', 0))
    fiber = safe_float(row.get('Fiber Content', 0))
    calories = safe_float(row.get('Calories', 0))
    sodium = safe_float(row.get('Sodium Content', 0))
    suitable_diabetes = int(safe_float(row.get('Suitable for Diabetes', 1)))
    suitable_bp = int(safe_float(row.get('Suitable for Blood Pressure', 1)))
    
    gi_map[food_name] = gi
    nutrition_map[food_name] = {
        'glycemic_index': gi,
        'calories': calories,
        'carbohydrates': carbs,
        'protein_g': protein,
        'fat_g': fat,
        'fiber_g': fiber,
        'sugar_g': round(carbs * 0.15, 1), 
        'sodium': sodium,
        'suitable_diabetes': suitable_diabetes,
        'suitable_bp': suitable_bp,
    }
    
    meal_image_map[food_name] = get_food_image_url(food_name)

print(f"Nutrition database: {len(nutrition_map)} foods")
print(f"\nSample entry (Apple):")
if 'Apple' in nutrition_map:
    for k, v in nutrition_map['Apple'].items():
        print(f"   {k}: {v}")

In [ ]:
print("\n[STEP 4] Categorizing Meals by Type...")
print("-" * 50)

MEAL_KEYWORDS = {
    'breakfast': ['oat', 'cereal', 'yogurt', 'egg', 'toast', 'pancake', 'milk', 
                  'paratha', 'idli', 'dosa', 'upma', 'poha', 'porridge', 'smoothie',
                  'muesli', 'granola', 'waffle'],
    'lunch': ['rice', 'chicken', 'fish', 'salad', 'sandwich', 'soup', 'pasta', 
              'quinoa', 'curry', 'dal', 'chapati', 'roti', 'biryani', 'wrap',
              'noodle', 'stir fry', 'bowl'],
    'dinner': ['salmon', 'steak', 'paneer', 'tofu', 'grilled', 'baked', 'tikka', 
               'kebab', 'roast', 'lamb', 'mutton', 'beef', 'turkey', 'shrimp'],
    'snacks': ['apple', 'banana', 'orange', 'berry', 'nut', 'almond', 'walnut', 
               'peanut', 'fruit', 'carrot', 'cucumber', 'hummus', 'chips', 'cracker']
}

def categorize_meal(food_name: str, nutrition: dict) -> str:
    """Categorize meal based on name and nutrition profile."""
    name_lower = food_name.lower()
    calories = nutrition['calories']
    protein = nutrition['protein_g']
    
    for meal_type, keywords in MEAL_KEYWORDS.items():
        if any(kw in name_lower for kw in keywords):
            if meal_type == 'snacks' and calories > 200:
                continue
            return meal_type
    
    if calories < 100:
        return 'snacks'
    elif protein >= 20:
        return 'dinner'
    elif calories < 300:
        return 'breakfast'
    else:
        return 'lunch'

meal_type_map = {}
type_counts = {'breakfast': 0, 'lunch': 0, 'dinner': 0, 'snacks': 0}

for food_name, nutrition in nutrition_map.items():
    meal_type = categorize_meal(food_name, nutrition)
    meal_type_map[food_name] = meal_type
    type_counts[meal_type] += 1

print("Meal Type Distribution:")
for mt, count in type_counts.items():
    print(f"   {mt.capitalize()}: {count} foods")

In [ ]:
print("\n[STEP 5] Generating Meal Descriptions...")
print("-" * 50)

def generate_description(food_name: str, nutrition: dict, meal_type: str) -> str:
    """Generate description from actual nutrition values."""
    parts = []
    gi = nutrition['glycemic_index']
    protein = nutrition['protein_g']
    fiber = nutrition['fiber_g']
    
    if gi > 0:
        if gi <= 35:
            parts.append("Low glycemic")
        elif gi <= 55:
            parts.append("Medium glycemic")
    
    if protein >= 20:
        parts.append("high-protein")
    elif protein >= 10:
        parts.append("protein-rich")
    
    if fiber >= 5:
        parts.append("fiber-rich")
    
    endings = {
        'breakfast': "for a balanced start to your day.",
        'lunch': "for sustained energy throughout the day.",
        'dinner': "for stable nighttime glucose levels.",
        'snacks': "to manage hunger between meals."
    }
    
    if parts:
        return f"A {', '.join(parts)} option {endings.get(meal_type, '')}"
    return f"A nutritious {meal_type} option {endings.get(meal_type, '')}"

meal_descriptions = {}
for food_name, nutrition in nutrition_map.items():
    meal_type = meal_type_map.get(food_name, 'lunch')
    meal_descriptions[food_name] = generate_description(food_name, nutrition, meal_type)

print(f"Generated {len(meal_descriptions)} descriptions")

In [ ]:
print("\n[STEP 6] Defining Diabetic-Safe Filtering Rules...")
print("-" * 50)

FILTER_RULES_BY_CONTROL = {
    'well controlled': {
        'max_glycemic_index': 60,
        'max_carbs_per_meal': 50,
        'max_sugar_per_meal': 12,
        'min_fiber': 2,
        'max_sodium': 700,
    },
    'moderately controlled': {
        'max_glycemic_index': 55,
        'max_carbs_per_meal': 45,
        'max_sugar_per_meal': 10,
        'min_fiber': 2,
        'max_sodium': 600,
    },
    'poorly controlled': {
        'max_glycemic_index': 40,
        'max_carbs_per_meal': 30,
        'max_sugar_per_meal': 6,
        'min_fiber': 3,
        'max_sodium': 500,
    }
}

GLUCOSE_ADJUSTMENTS = {
    'low': {'gi_penalty': 0, 'carb_penalty': 0},      
    'normal': {'gi_penalty': 0, 'carb_penalty': 0},   
    'high': {'gi_penalty': 10, 'carb_penalty': 10},    
    'very_high': {'gi_penalty': 20, 'carb_penalty': 15}
}

def get_glucose_category(glucose: float) -> str:
    if glucose < 100:
        return 'low'
    elif glucose < 140:
        return 'normal'
    elif glucose < 180:
        return 'high'
    else:
        return 'very_high'

print("Filter Rules by Control Level:")
for level, rules in FILTER_RULES_BY_CONTROL.items():
    print(f"\n{level.upper()}:")
    for rule, val in rules.items():
        print(f"   {rule}: {val}")

In [ ]:
print("\n[STEP 7] Calculating Safety Scores...")
print("-" * 50)

def calculate_safety_score(nutrition: dict, control_level: str, glucose: float) -> float:
    """Calculate diabetic safety score for a food item."""
    base_rules = FILTER_RULES_BY_CONTROL.get(control_level.lower(), 
                                              FILTER_RULES_BY_CONTROL['moderately controlled'])
    glucose_cat = get_glucose_category(glucose)
    adjustments = GLUCOSE_ADJUSTMENTS[glucose_cat]
    
    max_gi = base_rules['max_glycemic_index'] - adjustments['gi_penalty']
    max_carbs = base_rules['max_carbs_per_meal'] - adjustments['carb_penalty']
    
    score = 100
    
    if nutrition['glycemic_index'] > max_gi:
        score -= 25
    elif nutrition['glycemic_index'] <= 35:
        score += 5 
    
    if nutrition['carbohydrates'] > max_carbs:
        score -= 20
    
    if nutrition.get('sugar_g', 0) > base_rules['max_sugar_per_meal']:
        score -= 15
    
    if nutrition['fiber_g'] >= base_rules['min_fiber']:
        score += 5
    
    if nutrition['protein_g'] >= 10:
        score += 5
    
    if nutrition.get('suitable_diabetes', 1) == 0:
        score -= 30
    
    return max(0, min(100, score))

default_scores = {}
for food_name, nutrition in nutrition_map.items():
    default_scores[food_name] = calculate_safety_score(
        nutrition, 'moderately controlled', 130
    )

safe_count = sum(1 for s in default_scores.values() if s >= 70)
print(f"Foods with safety score >= 70: {safe_count} / {len(nutrition_map)}")

In [ ]:
print("\n[STEP 8] Extracting Training Data...")
print("-" * 50)

glucose_values = diabetes_patient_df['Fasting_Blood_Glucose'].dropna().values
glucose_values = glucose_values[(glucose_values >= 50) & (glucose_values <= 350)]

hba1c_values = diabetes_patient_df['HbA1c'].dropna().values
hba1c_values = hba1c_values[(hba1c_values >= 4) & (hba1c_values <= 14)]

bmi_values = diabetes_patient_df['BMI'].dropna().values
bmi_values = bmi_values[(bmi_values >= 15) & (bmi_values <= 50)]

print(f"Blood Glucose samples: {len(glucose_values)}")
print(f"   Range: {glucose_values.min():.0f} - {glucose_values.max():.0f} mg/dL")
print(f"   Mean: {glucose_values.mean():.0f} mg/dL")

print(f"\nHbA1c samples: {len(hba1c_values)}")
print(f"   Range: {hba1c_values.min():.1f} - {hba1c_values.max():.1f}%")
print(f"   Mean: {hba1c_values.mean():.1f}%")

print(f"\nBMI samples: {len(bmi_values)}")
print(f"   Range: {bmi_values.min():.1f} - {bmi_values.max():.1f}")
print(f"   Mean: {bmi_values.mean():.1f}")

In [ ]:
print("\nMapping HbA1c to Control Levels:")

def hba1c_to_control_level(hba1c: float) -> str:
    """Map HbA1c to diabetes control level."""
    if hba1c < 7.0:
        return 'well controlled'
    elif hba1c < 8.5:
        return 'moderately controlled'
    else:
        return 'poorly controlled'

# Count distribution
control_counts = {'well controlled': 0, 'moderately controlled': 0, 'poorly controlled': 0}
for hba1c in hba1c_values:
    control_counts[hba1c_to_control_level(hba1c)] += 1

print("   HbA1c < 7.0% → Well controlled")
print("   HbA1c 7.0-8.5% → Moderately controlled")
print("   HbA1c > 8.5% → Poorly controlled")
print(f"\nDistribution in dataset:")
for level, count in control_counts.items():
    print(f"   {level}: {count} ({count/len(hba1c_values)*100:.1f}%)")

In [ ]:
print("\n[STEP 9] Generating Training Dataset...")
print("-" * 50)

np.random.seed(42)

X_data = []
y_data = []

for food_name, nutrition in nutrition_map.items():
    meal_type = meal_type_map.get(food_name, 'lunch')
    gi = nutrition['glycemic_index']
    
    for control_level, control_code in CONTROL_LEVEL_ENCODING.items():
        avg_glucose = 120 if control_level == 'well controlled' else (150 if control_level == 'moderately controlled' else 180)
        base_score = calculate_safety_score(nutrition, control_level, avg_glucose)
        
        if base_score < 60: 
            continue
        
        n_samples = max(5, int(base_score / 10))
        
        if gi <= 35:
            glucose_range = (80, 300) 
        elif gi <= 55:
            glucose_range = (80, 180) 
        else:
            glucose_range = (70, 140)
        
        suitable_glucose = glucose_values[
            (glucose_values >= glucose_range[0]) & (glucose_values <= glucose_range[1])
        ]
        if len(suitable_glucose) < 10:
            suitable_glucose = glucose_values
        
        for portion_size, portion_code in PORTION_SIZE_ENCODING.items():
            sampled_glucose = np.random.choice(
                suitable_glucose, 
                size=min(n_samples, len(suitable_glucose)), 
                replace=True
            )
            
            for glucose in sampled_glucose:
                if control_level == 'well controlled':
                    hba1c = np.random.uniform(5.0, 6.9)
                elif control_level == 'moderately controlled':
                    hba1c = np.random.uniform(7.0, 8.4)
                else:
                    hba1c = np.random.uniform(8.5, 12.0)
                
                X_data.append([
                    float(glucose),
                    float(hba1c),
                    control_code,
                    portion_code,
                    MEAL_TYPE_ENCODING[meal_type]
                ])
                y_data.append(food_name)

X = np.array(X_data)
y = np.array(y_data)

print(f"Training samples generated: {len(X):,}")
print(f"Unique foods: {len(set(y))}")
print(f"\nFeatures: [glucose, hba1c, control_level, portion_size, meal_type]")
print(f"Feature shape: {X.shape}")

In [ ]:
print("\n[STEP 10] Training Model...")
print("-" * 50)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(
    n_neighbors=15,
    weights='distance',
    metric='euclidean',
    n_jobs=-1
)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model: K-Nearest Neighbors (k=15)")
print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"Test accuracy: {accuracy:.3f}")

print("\nMeal types in trained model:")
for mt in ['breakfast', 'lunch', 'dinner', 'snacks']:
    count = sum(1 for name in label_encoder.classes_ if meal_type_map.get(name) == mt)
    print(f"   {mt}: {count} foods")

In [ ]:
print("\n[STEP 11] Testing Recommendation Function...")
print("-" * 50)

def get_recommendations(
    current_glucose: float,
    diabetes_control_level: str,
    portion_size: str,
    time: str,
    n_per_type: int = 4
) -> dict:
    """Get meal recommendations based on user inputs."""
    
    control_code = CONTROL_LEVEL_ENCODING.get(diabetes_control_level.lower(), 1)
    portion_code = PORTION_SIZE_ENCODING.get(portion_size.lower(), 1)
    meal_type_from_time = time_to_meal_type(time)
    
    hba1c_map = {'well controlled': 6.0, 'moderately controlled': 7.5, 'poorly controlled': 9.0}
    hba1c = hba1c_map.get(diabetes_control_level.lower(), 7.5)
    
    portion_mult = PORTION_MULTIPLIERS.get(portion_size.lower(), 1.0)
    
    recommendations = {'breakfast': [], 'lunch': [], 'dinner': [], 'snacks': []}
    
    for meal_type, type_code in MEAL_TYPE_ENCODING.items():
        features = np.array([[current_glucose, hba1c, control_code, portion_code, type_code]])
        features_scaled = scaler.transform(features)
        
        proba = knn.predict_proba(features_scaled)[0]
        top_indices = np.argsort(proba)[::-1]
        
        count = 0
        for idx in top_indices:
            if count >= n_per_type:
                break
            
            food_name = label_encoder.inverse_transform([idx])[0]
            
            if meal_type_map.get(food_name) != meal_type:
                continue
            
            score = calculate_safety_score(
                nutrition_map[food_name], 
                diabetes_control_level, 
                current_glucose
            )
            if score < 60:
                continue
            
            nutrition = nutrition_map[food_name]
            
            recommendations[meal_type].append({
                'name': food_name,
                'description': meal_descriptions.get(food_name, ''),
                'glycemic_index': int(nutrition['glycemic_index']),
                'image_url': meal_image_map.get(food_name, ''),
                'nutrition_facts': {
                    'proteins_g': round(nutrition['protein_g'] * portion_mult, 1),
                    'carbohydrates_g': round(nutrition['carbohydrates'] * portion_mult, 1),
                    'fats_g': round(nutrition['fat_g'] * portion_mult, 1),
                    'sugar_g': round(nutrition.get('sugar_g', 0) * portion_mult, 1),
                    'fiber_g': round(nutrition['fiber_g'] * portion_mult, 1)
                },
                'safety_score': score
            })
            count += 1
    
    return recommendations

test_input = {
    "current_glucose": 130,
    "diabetes_control_level": "Moderately controlled",
    "portion_size": "Medium",
    "time": "12:30"
}

print(f"Test Input: {test_input}")
print(f"Detected meal type: {time_to_meal_type(test_input['time'])}")

test_recs = get_recommendations(**test_input)

print("\nRecommendations:")
for meal_type, meals in test_recs.items():
    print(f"\n{meal_type.upper()}:")
    for m in meals[:2]:
        print(f"   - {m['name']} (GI: {m['glycemic_index']}, Score: {m['safety_score']})")

In [ ]:
print("\n[STEP 12] Exporting Model...")
print("-" * 50)

export = {
    'model': knn,
    'scaler': scaler,
    'label_encoder': label_encoder,
    
    'feature_names': ['current_glucose', 'hba1c', 'control_level', 'portion_size', 'meal_type'],
    
    'control_level_encoding': CONTROL_LEVEL_ENCODING,
    'portion_size_encoding': PORTION_SIZE_ENCODING,
    'meal_type_encoding': MEAL_TYPE_ENCODING,
    'portion_multipliers': PORTION_MULTIPLIERS,
    
    'filter_rules_by_control': FILTER_RULES_BY_CONTROL,
    'glucose_adjustments': GLUCOSE_ADJUSTMENTS,
    
    'gi_map': gi_map,
    'nutrition_map': nutrition_map,
    'meal_type_map': meal_type_map,
    'meal_descriptions': meal_descriptions,
    'meal_image_map': meal_image_map,
    'meal_scores': default_scores,
    
    'total_meals': len(nutrition_map),
    'training_samples': len(X_train),
    'test_accuracy': accuracy,
}

output_path = OUTPUT_DIR / 'meal_recommendations_trained.pkl'
joblib.dump(export, output_path)

print(f"Model exported successfully!")
print(f"\nExport Summary:")
print(f"   Path: {output_path}")
print(f"   Total meals: {len(nutrition_map)}")
print(f"   Training samples: {len(X_train):,}")
print(f"   Test accuracy: {accuracy:.3f}")
print(f"\nFeatures used:")
print(f"   - current_glucose (from user input)")
print(f"   - hba1c (derived from control level)")
print(f"   - control_level (Well/Moderately/Poorly controlled)")
print(f"   - portion_size (Small/Medium/Large)")
print(f"   - meal_type (from time of day)")

In [ ]:
import json

print("\n" + "=" * 70)
print("SAMPLE API REQUEST & RESPONSE")
print("=" * 70)

sample_request = {
    "current_glucose": 130,
    "diabetes_control_level": "Moderately controlled",
    "portion_size": "Medium",
    "time": "12:30",
    "user_id": "7a22a0a0-67bf-41ab-9e60-393b5ac2e7df"
}

print("\nRequest:")
print(json.dumps(sample_request, indent=2))

recs = get_recommendations(
    current_glucose=sample_request['current_glucose'],
    diabetes_control_level=sample_request['diabetes_control_level'],
    portion_size=sample_request['portion_size'],
    time=sample_request['time']
)

api_response = {
    "success": True,
    "message": "Meal recommendations generated successfully",
    "data": {
        "status": "success",
        "detected_meal_type": time_to_meal_type(sample_request['time']),
        "recommendations": recs
    }
}

print("\nResponse:")
print(json.dumps(api_response, indent=2, default=str)[:3000])
print("...")

In [ ]:
print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"\nModel saved to: {output_path}")
print(f"\nThe model uses:")
print(f"   - {len(nutrition_map)} foods from meal_recommender.csv")
print(f"   - {len(glucose_values)} glucose samples from diabetes_dataset.csv")
print(f"   - {len(hba1c_values)} HbA1c samples from diabetes_dataset.csv")
print(f"\nRecommendations are personalized based on:")
print(f"   - Current glucose level")
print(f"   - Diabetes control level (HbA1c-based)")
print(f"   - Portion size preference")
print(f"   - Time of day (meal type)")